In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os
import socket
from dataclasses import dataclass
from typing import Any

In [3]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()
print(f"Project Root: {project_root}")
ROOT_DATASET_DIR = project_root / "dataset" 

Project Root: /storage/ice1/1/0/vchopra37/projects/learning_env


In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [5]:
import tiktoken 
import torch
import torch.nn as nn
from torch.nn import functional as F 

In [6]:
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

In [7]:
if IS_PACE_SERVER:
    from model_training.artifacts.model.load_model.gpt2.configs import load_model_config
    from model_training.artifacts.model.load_model.gpt2.model import GPT
    from model_training.artifacts.model.components.inference.inference import gen_sample
    from model_training.artifacts.data.pre_train_dataset.tinyshakespeare.data import return_data, prepare_one_batch, DataLoaderLite
    from model_training.artifacts.train.tracking.logger import ExperimentLogger
    from model_training.artifacts.train.core.train import train_model
    from model_training.artifacts.train.core.config import TrainConfig
    from model_training.artifacts.train.utils.optimizer import configure_optimizer
    from model_training.artifacts.train.utils.lr_scheduler import cosine_lr
else:
    ## Fix this import
    from model_training.artifacts.model.load_model.gpt2.model import GPT
        # from src.model import GPT

In [8]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [9]:
@dataclass
class TrainComponents:
    optimizer:Any
    dataloader:DataLoaderLite
    scheduler:Any
    
tokenizer = tiktoken.get_encoding("gpt2")
total_batch_size = 524288 ## 0.5 M (gpt3 paper)


micro_batch_size= 8
seq_length = 1024

assert total_batch_size % (micro_batch_size * seq_length) ==0 , "Make sure total batch size is divisible"
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length)

config = load_model_config(seq_length, "gpt2-small")
config.vocab_size = 50304           # Small Change to Vocab Size, Numbers in the power of 2 
config

loading wights from pretrained gpt: gpt2-small
{'n_layer': 12, 'n_head': 12, 'emb_dim': 768}


GPT2Config(context_len=1024, vocab_size=50304, n_layer=12, n_head=12, emb_dim=768)

#### Step-1:- Load the dataset

In [10]:
dataloader = DataLoaderLite(batch_size=micro_batch_size, seq_len=seq_length, tokenizer=tokenizer, dataset_dir=ROOT_DATASET_DIR)

Loaded 338025 tokens
1 epoch = 41 tokens


In [11]:

len(dataloader.enc_data) // (dataloader.seq_len)

330

### Step-2:- Load the Model 

In [12]:
model = GPT(config=config)
print(f"Model Loaded: {model}")
model.eval()            # Putting it in eval mode

Model Loaded: GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50304, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50304, bias=False)
)


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50304, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50304, bias=False)
)

### Step-3:- Training the Model 

In [13]:

train_config = TrainConfig(num_steps=50, 
                        learning_rate=3e-4, 
                        total_batch_size= total_batch_size,
                        micro_batch_size =micro_batch_size, 
                        seq_length = seq_length, 
                        max_lr = 6e-4, 
                        min_lr =  6e-4 * 0.1,
                        warmup_steps= 715, 
                        accumulation_steps = grad_accum_steps,
                        # max_steps = 19073   # 19,073 steps is ~1 epoch, if data is 10B tokens and batch size 0.5M tokens
                        device=device)


In [14]:
optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

num decayed parameter tensors: 50, with 124,354,560 parameters
num non-decayed parameter tensors: 98, with 121,344 parameters
using fused AdamW: True


In [15]:
train_comp = TrainComponents(optimizer=optimizer , dataloader=dataloader, scheduler= cosine_lr)
train_comp

TrainComponents(optimizer=AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 0.0003
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 0.0003
    maximize: False
    weight_decay: 0.0
), dataloader=<model_training.artifacts.data.pre_train_dataset.tinyshakespeare.data.DataLoaderLite object at 0x155200ff4c50>, scheduler=<function cosine_lr at 0x155201016ac0>)

In [16]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [ ]:
%timeit
### Training on Multiple Batches
model = train_model(model, train_components=train_comp, train_config=train_config)